In [5]:
import pandas as pd
import numpy as np

# Import training and test data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# DATES FEATURES
def date_features(df):
    # Date Features
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df.date.dt.year
    df['month'] = df.date.dt.month
    df['day'] = df.date.dt.day
    df['dayofyear'] = df.date.dt.dayofyear
    df['dayofweek'] = df.date.dt.dayofweek
    df['weekofyear'] = df.date.dt.isocalendar().week
    
    # Additionnal Data Features
    df['day^year'] = np.log((np.log(df['dayofyear'] + 1)) ** (df['year'] - 2000))
    
    # Drop date
    df.drop('date', axis=1, inplace=True)
    
    return df

# Merge Test with Daily Avg, Monthly Avg
def merge(df1, df2, col,col_name):
    
    df1 =pd.merge(df1, df2, how='left', on=None, left_on=col, right_on=col,
            left_index=False, right_index=False, sort=True,
             copy=True, indicator=False)
    
    df1 = df1.rename(columns={'sales':col_name})
    return df1


In [6]:
# Dates Features for Train, Test
train, test = date_features(train), date_features(test)

train

,store,item,sales,year,month,day,dayofyear,dayofweek,weekofyear,day^year
0,1,1,13,2013,1,1,1,1,1,-4.764668
1,1,1,11,2013,1,2,2,2,1,1.222622
2,1,1,14,2013,1,3,3,3,1,4.246245
3,1,1,13,2013,1,4,4,4,1,6.186505
4,1,1,10,2013,1,5,5,5,1,7.581575
...,...,...,...,...,...,...,...,...,...,...
912995,10,50,63,2017,12,27,361,2,52,30.150097
912996,10,50,59,2017,12,28,362,3,52,30.158055
912997,10,50,74,2017,12,29,363,4,52,30.165987
912998,10,50,62,2017,12,30,364,5,52,30.173894


In [4]:
# Daily Average, Monthly Average for train
train['daily_avg']  = train.groupby(['item','store','dayofweek'])['sales'].transform('mean')
train['monthly_avg'] = train.groupby(['item','store','month'])['sales'].transform('mean')
train = train.dropna()

# Average sales for Day_of_week = d per Item,Store
daymonth_avg = train.groupby(['item','store','dayofweek'])['sales'].mean().reset_index()
# Average sales for Month = m per Item,Store
monthly_avg = train.groupby(['item','store','month'])['sales'].mean().reset_index()

In [7]:
# Add Daily_avg and Monthly_avg features to test 
test = merge(test, daymonth_avg,['item','store','dayofweek'],'daily_avg')
test = merge(test, monthly_avg,['item','store','month'],'monthly_avg')

# Sales Rolling mean sequence per item 
rolling_10 = train.groupby(['item'])['sales'].rolling(10).mean().reset_index().drop('level_1', axis=1)
train['rolling_mean'] = rolling_10['sales'] 

# 90 last days of training rolling mean sequence added to test data
rolling_last90 = train.groupby(['item','store'])['rolling_mean'].tail(90).copy()
test['rolling_mean'] = rolling_last90.reset_index().drop('index', axis=1)

# Shifting rolling mean 3 months
train['rolling_mean'] = train.groupby(['item'])['rolling_mean'].shift(90) # Create a feature with rolling mean of day - 90
train.head()

,store,item,sales,year,month,day,dayofyear,dayofweek,weekofyear,day^year,rolling_mean
0,1,1,13,2013,1,1,1,1,1,-4.764668,NaN
1,1,1,11,2013,1,2,2,2,1,1.222622,NaN
2,1,1,14,2013,1,3,3,3,1,4.246245,NaN
3,1,1,13,2013,1,4,4,4,1,6.186505,NaN
4,1,1,10,2013,1,5,5,5,1,7.581575,NaN


In [8]:
# Clean features highly correlated to each others
for df in [train, test]:
    df.drop(['dayofyear', 
                  'weekofyear',
                  'daily_avg',
                  'day',
                  'month',
                  'item',
                  'store',],
                 axis=1, 
                 inplace=True)
    
# Features Scaling (except sales)
sales_series, id_series = train['sales'], test['id']
# Features Scaling
train = (train - train.mean()) / train.std()
test = (test - test.mean()) / test.std()
# Retrieve actual Sales values and ID
train['sales'] = sales_series
test['id'] = id_series

# Training Data
X_train = train.drop('sales', axis=1).dropna()
y_train = train['sales']
# Test Data
test.sort_values(by=['id'], inplace=True)
X_test = test.drop('id', axis=1)
#df = train
df_train = train.copy()

# Train Test Split
X_train , X_test ,y_train, y_test = train_test_split(df_train.drop('sales',axis=1),df_train.pop('sales'), random_state=123, test_size=0.2)

# XGB Model
matrix_train = xgb.DMatrix(X_train, label = y_train)
matrix_test = xgb.DMatrix(X_test, label = y_test)

# Run XGB 
model = xgb.train(params={'objective':'reg:linear','eval_metric':'mae'}
                ,dtrain = matrix_train, num_boost_round = 500, 
                early_stopping_rounds = 20, evals = [(matrix_test,'test')],)

KeyError: "['daily_avg'] not found in axis"